In [ ]:
import requests, re
from bs4 import BeautifulSoup
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

# 1. Get web pages
urls = ["https://www.bbc.com/news/technology",
        "https://www.cnn.com/tech"]

def scrape(url):
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        r.raise_for_status()
        return " ".join(p.get_text(" ", strip=True)
                        for p in BeautifulSoup(r.text, "html.parser").find_all("p"))
    except requests.RequestException:
        print("Could not access:", url)
        return ""

docs = [scrape(u) for u in urls]
docs = [d for d in docs if d.strip()]

if len(docs) < 2:
    raise ValueError("Could not scrape enough pages.")

# 2. Clean text
def clean(text):
    words = re.findall(r"\b[a-z]+\b", text.lower())
    return " ".join(w for w in words
                    if w not in ENGLISH_STOP_WORDS and len(w) > 2)

docs = [clean(d) for d in docs]

# 3. Convert text into TF-IDF
v = TfidfVectorizer(max_df=1.0, min_df=1)
X = v.fit_transform(docs)
words = v.get_feature_names_out()

# 4. Apply NMF
n = min(3, X.shape[0], X.shape[1])
model = NMF(n_components=n, random_state=42, max_iter=500)
model.fit(X)

# 5. Display topics
print("\n--- Topics Identified ---")
for i, topic in enumerate(model.components_):
    top = topic.argsort()[-10:][::-1]
    print(f"\nTopic {i+1}:")
    print(", ".join(words[top]))


--- Topics Identified ---

Topic 1:
jones, dow, indices, llc, market, cnn, provided, reserved, news, rights

Topic 2:
tech, bbc, hack, data, binance, believed, backlash, online, beijing, world
